# 02. Classifier Benchmark — TRTR / TSTR / TSRTR / TATR

Trains and cross-validates the four benchmark classifiers (Apollo RF+SVM, Random
Forest, XGBoost, 1D-CNN) on purified tumour spectra under a selectable **training
regimen**:

| Regimen   | Train on                          | Test on          |
|-----------|-----------------------------------|------------------|
| **TRTR**  | real (slide-capped)               | real validation  |
| **TSTR**  | CVAE synthetic pool               | real validation  |
| **TSRTR** | real + synthetic                  | real validation  |
| **TATR**  | real + on-the-fly augmentation    | real validation  |

Set `CONFIG_NAME` and `REGIMEN` in the setup cell below, then run top-to-bottom.

### Pipeline parity
The per-fold preparation and the 4-model training step are imported from
`pipeline/_common.py` — the *same* helpers the headless suite uses
(`python -m pipeline.run_suite`). This notebook is a thin, inspectable driver
over that shared code, so the notebook and the pipeline cannot drift. Production
runs invoke the stage modules directly — see [`pipeline/README.md`](../pipeline/README.md).

### Per-fold preparation (`prepare_fold`)
1. Labels normalised to 0-indexed.
2. Binary-task mapping when `models.cnn.num_classes == 2` (raw class < 3 → 0 *Mutant*, else → 1 *Wildtype*).
3. **Slide-level cap** applied to the **training** split only: at most
   `cv_strategy.patient_cap` spectra per slide, so slides contributing many
   correlated spectra don't dominate training. The validation split is never capped. Here (classifier stages) the cap is a **single fixed subsample** drawn once per fold; the CVAE stage (notebook 03) instead re-samples it fresh each epoch.

**Inputs:** `paths.processed_file` (H5) + `paths.splits_file`, both from the config.
**Outputs:** `apollo_{fold}.joblib`, `rf_{fold}.joblib`, `xgb_{fold}.json`,
`cnn_{fold}.keras` under the selected regimen's model directory.

> **Notes for reviewers.** `TATR` is notebook-only (it is *not* part of the
> reproducible suite). `TSTR`/`TSRTR` here load a **pre-generated** synthetic pool
> (`{fold}_X_synth.npy`, produced by notebook 03 / the `cvae` stage). The
> production `TSRTR` instead generates synthetics *on demand* at an exact
> real:synth ratio in `pipeline/tsrtr_stage.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Project root on path (this notebook lives in notebooks/). Must precede the
# `src` / `pipeline` imports below.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebooks.experiment_utils import setup_experiment
from pipeline._common import (
    make_loader, prepare_fold, train_four_models,
    set_seeds, print_benchmark_summary,
)
from pipeline.tstr_stage import _load_synth_pool   # shared synthetic-pool loader

# ── Experiment selection ───────────────────────────────────────────────
CONFIG_NAME = 'exp_608'    # any configs/exp_*.yaml
REGIMEN     = 'TRTR'       # TRTR | TSTR | TSRTR | TATR  (see table above)

# setup_experiment() also chdir's to the project root so the config's relative
# paths resolve regardless of where the kernel started.
config, output_dir = setup_experiment(CONFIG_NAME)
seed = set_seeds(config)

# Model output directory per regimen
MODEL_DIRS = {
    'TRTR':  config['paths'].get('models_baseline_dir') or config['paths']['models_trtr_dir'],
    'TSTR':  config['paths']['models_tstr_dir'],
    'TSRTR': config['paths']['models_tsrtr_dir'],
    'TATR':  config['paths'].get('models_tatr_dir', f"results/{CONFIG_NAME}/models/tatr"),
}
model_dir = Path(MODEL_DIRS[REGIMEN])
model_dir.mkdir(parents=True, exist_ok=True)

print(f"Experiment : {CONFIG_NAME}  ({config.get('description', '')})")
print(f"Regimen    : {REGIMEN}")
print(f"Seed       : {seed}")
print(f"Models ->  : {model_dir}")

## Run the cross-validated benchmark

The regimen switchboard decides *what to train on*; `train_four_models` (shared with the pipeline) does the training and saves one model per classifier per fold.

In [ ]:
loader = make_loader(config)
print(f"Dataset: {loader.X.shape[0]} spectra x {loader.X.shape[1]} features | "
      f"{len(np.unique(loader.groups))} slides | "
      f"{len(np.unique(loader.y_class))} raw classes | {len(loader)} folds")

synth_dir = Path(config['paths']['synthetic_dir'])
per_fold = []

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n--- {fold_name} ---")

    # Capped real training data + (never-capped) real validation. The cap is only
    # meaningful for regimens that train on real spectra.
    cap = REGIMEN in ('TRTR', 'TSRTR', 'TATR')
    X_real, y_real, X_val, y_val = prepare_fold(fold_data, config, cap_train=cap)

    # ── Regimen switchboard: build the TRAINING set ───────────────────────
    if REGIMEN == 'TRTR':
        X_train, y_train = X_real, y_real

    elif REGIMEN == 'TSTR':
        X_train, y_train = _load_synth_pool(synth_dir, fold_name)

    elif REGIMEN == 'TSRTR':
        X_syn, y_syn = _load_synth_pool(synth_dir, fold_name)
        X_train = np.vstack([X_real, X_syn])
        y_train = np.concatenate([y_real, y_syn])
        perm = np.random.permutation(len(y_train))
        X_train, y_train = X_train[perm], y_train[perm]

    elif REGIMEN == 'TATR':
        from src.data.augmentation import AugmentedDataGenerator
        aug_cfg = config.get('augmentation', {})
        if aug_cfg.get('enabled', False):
            gen = AugmentedDataGenerator(X_real, y_real, aug_cfg)
            batches = list(gen.get_batches(batch_size=256, epochs=1, shuffle=True))
            X_aug = np.vstack([bx for bx, _ in batches])
            y_aug = np.concatenate([by for _, by in batches])
            X_train = np.vstack([X_real, X_aug])
            y_train = np.concatenate([y_real, y_aug])
            perm = np.random.permutation(len(y_train))
            X_train, y_train = X_train[perm], y_train[perm]
        else:
            print("  Augmentation disabled in config -> training on real data only")
            X_train, y_train = X_real, y_real

    else:
        raise ValueError(f"Unknown REGIMEN: {REGIMEN!r}")

    accs = train_four_models(X_train, y_train, X_val, y_val,
                             config, model_dir, fold_name, seed)
    per_fold.append(accs)

print_benchmark_summary(per_fold, title=f"{REGIMEN} BENCHMARK COMPLETE")

## Results — mean validation accuracy per model

In [ ]:
models = ['Apollo', 'RF_Standard', 'XGBoost', '1D_CNN']
means = [np.nanmean([d.get(m, np.nan) for d in per_fold]) for m in models]
stds  = [np.nanstd([d.get(m, np.nan) for d in per_fold]) for m in models]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(models, means, yerr=stds, capsize=5, color='#4c72b0', alpha=0.85)
for i, (m, s) in enumerate(zip(means, stds)):
    if not np.isnan(m):
        ax.text(i, m + (s if not np.isnan(s) else 0) + 0.01, f"{m:.3f}",
                ha='center', fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Validation accuracy')
ax.set_title(f"{CONFIG_NAME} - {REGIMEN}  ({len(per_fold)}-fold CV, mean +/- std)")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()